# Notebook 10 — StratLake Walk-Forward Robustness and Promotion Review

This notebook is a Colab-first source notebook for Notebook 10. It composes the restored Notebook 08/09 StratLake archive with native strategy execution, walk-forward smoke/expanded windows, robustness diagnostics, and promotion-gate review.

The imported source preserves the successful smoke-test behavior and keeps the warning-category taxonomy clear so numeric, benchmark-degenerate, strategy-degenerate, QA, signal-consistency, missing-contract, and runtime-error warnings are distinguishable in the audit artifacts.

**Important:** smoke mode validates workflow wiring and diagnostic surfacing. It is not promotion-grade financial evidence. Use expanded mode before interpreting promotion outcomes as research evidence.


## 1. Install notebook dependencies and app packages

Keep this aligned with Notebook 08/09 while staging. The first pass should validate the same public/test package installation pathway used by prior notebooks.


In [ ]:
!pip install "pandas-market-calendars>=5.0"
!pip install -i https://test.pypi.org/simple/ fintech-market-ingestion
!pip install -i https://test.pypi.org/simple/ stratlake-trade-engine


## 2. Imports, Colab detection, and Google Drive auth


In [ ]:
from pathlib import Path
import os
import re
import sys
import json
import shutil
import subprocess
import getpass
from datetime import datetime, timezone
from typing import Any

import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    drive = None
    IN_COLAB = False

print("IN_COLAB:", IN_COLAB)
print("Python executable:", sys.executable)
print("Current working directory:", Path.cwd().as_posix())

if IN_COLAB:
    drive.mount("/content/drive")
    print("Google Drive mounted.")
else:
    print("Not running in Colab; skipping Google Drive mount.")


## 3. Load Alpaca environment variables

Notebook 10 primarily restores existing Notebook 08/09 data. The credentials are still loaded so native commands that validate the environment behave consistently with earlier notebooks.


In [ ]:
try:
    from google.colab import userdata
except Exception:
    userdata = None

def get_secret_or_prompt(name: str) -> str:
    value = None
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value:
        value = getpass.getpass(f"Enter {name}: ")
    return value

alpaca_api_key_id = get_secret_or_prompt("ALPACA_API_KEY_ID")
alpaca_api_secret_key = get_secret_or_prompt("ALPACA_API_SECRET_KEY")

if not alpaca_api_key_id or not alpaca_api_secret_key:
    raise ValueError("Missing Alpaca API credentials.")

os.environ["ALPACA_API_KEY_ID"] = alpaca_api_key_id
os.environ["ALPACA_API_SECRET_KEY"] = alpaca_api_secret_key
os.environ["ALPACA_DATA_BASE_URL"] = "https://data.alpaca.markets"
os.environ["ALPACA_FEED"] = "iex"

print("ALPACA_DATA_BASE_URL:", os.environ.get("ALPACA_DATA_BASE_URL"))
print("ALPACA_FEED:", os.environ.get("ALPACA_FEED"))
print("ALPACA_API_KEY_ID and ALPACA_API_SECRET_KEY are set but not printed.")


## 4. Configure workspace, sessions, archive paths, and notebook mode

The default mode is intentionally a **smoke test**. It validates runtime wiring, archive restore, strategy preflight, native execution, artifact parsing, diagnostics, and promotion-gate behavior. Smoke mode should not be treated as promotion-grade financial evidence.

After the one-window smoke test succeeds, set `NOTEBOOK10_MODE = "expanded"` and rerun before using the notebook for research interpretation.


In [ ]:
WORKSPACE_ROOT = Path("/content") if IN_COLAB else Path.cwd()

# Keep this aligned with Notebook 08/09. Replace the placeholder before any Drive restore or archive action.
DRIVE_FOLDER_NAME = "REPLACE_WITH_DRIVE_FOLDER_NAME"
DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER_NAME if IN_COLAB else WORKSPACE_ROOT / "drive" / DRIVE_FOLDER_NAME

if DRIVE_FOLDER_NAME == "REPLACE_WITH_DRIVE_FOLDER_NAME":
    raise ValueError("Set DRIVE_FOLDER_NAME before creating Google Drive session/archive folders.")

FINTECH_ROOT = WORKSPACE_ROOT / "fintech-market-ingestion-demo"
STRATLAKE_ROOT = WORKSPACE_ROOT / "stratlake-trade-engine-demo"

FINTECH_DRIVE_ROOT = DRIVE_ROOT / "fintech-market-ingestion"
STRATLAKE_DRIVE_ROOT = DRIVE_ROOT / "stratlake-trade-engine"

FINTECH_SESSION_NAME = "fintech_stratlake_input"
STRATLAKE_SESSION_NAME = "stratlake_q1_feature_consumption"

FINTECH_SESSION_ID_OVERRIDE = ""
STRATLAKE_SESSION_ID_OVERRIDE = "stratlake_q1_feature_consumption"

ANALYSIS_START = "2026-01-02"
ANALYSIS_END = "2026-03-31"

BACKFILL_START = "2025-11-03"
BACKFILL_END = "2026-04-15"
FEATURE_BUILD_START = BACKFILL_START
FEATURE_BUILD_END = BACKFILL_END
BACKFILL_SYMBOLS = "AAPL,MSFT,NVDA,SPY,QQQ"

# Modes: smoke validates wiring; expanded gives better but still preliminary robustness evidence.
NOTEBOOK10_MODE = "smoke"  # "smoke" or "expanded"
RUN_NATIVE_WALK_FORWARD_EVALUATION = True
RUN_ONLY_PREFLIGHT_RUNNABLE_STRATEGIES = True
CANDIDATE_STRATEGY_LIMIT = None  # e.g. 5 for a smaller first pass; None runs all discovered runnable candidates.

SMOKE_WINDOWS = [
    {"window_name": "smoke_2026_01", "start": "2026-01-02", "end": "2026-01-31"},
]

EXPANDED_WALK_FORWARD_WINDOWS = [
    {"window_name": "q1_full", "start": "2026-01-02", "end": "2026-03-31"},
    {"window_name": "jan_2026", "start": "2026-01-02", "end": "2026-01-31"},
    {"window_name": "feb_2026", "start": "2026-02-02", "end": "2026-02-27"},
    {"window_name": "mar_2026", "start": "2026-03-02", "end": "2026-03-31"},
    {"window_name": "rolling_45d_01", "start": "2026-01-02", "end": "2026-02-17"},
    {"window_name": "rolling_45d_02", "start": "2026-02-18", "end": "2026-03-31"},
]

WALK_FORWARD_WINDOWS = SMOKE_WINDOWS if NOTEBOOK10_MODE == "smoke" else EXPANDED_WALK_FORWARD_WINDOWS

# Leave empty to use detected strategies from configs/strategies.yml.
CANDIDATE_STRATEGIES_OVERRIDE = []

for path in [FINTECH_ROOT, STRATLAKE_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print("WORKSPACE_ROOT:", WORKSPACE_ROOT.as_posix())
print("DRIVE_ROOT:", DRIVE_ROOT.as_posix())
print("FINTECH_ROOT:", FINTECH_ROOT.as_posix())
print("STRATLAKE_ROOT:", STRATLAKE_ROOT.as_posix())
print("NOTEBOOK10_MODE:", NOTEBOOK10_MODE)
print("Walk-forward windows:", len(WALK_FORWARD_WINDOWS))
if NOTEBOOK10_MODE == "smoke":
    print("SMOKE MODE WARNING: results validate workflow wiring only; they are not promotion-grade financial evidence.")
display(pd.DataFrame(WALK_FORWARD_WINDOWS))


## 5. Verify installed native CLI commands and import surfaces


In [ ]:
cli_commands = [
    "fintech-init-project",
    "fintech-backfill-daily",
    "fintech-backup-data",
    "stratlake-init-session",
    "stratlake-run-strategy",
    "stratlake-session-archive-bootstrap",
    "stratlake-session-archive-restore-bootstrap",
]

cli_status_rows = []
for cmd in cli_commands:
    result = subprocess.run([cmd, "--help"], text=True, capture_output=True)
    cli_status_rows.append({
        "command": cmd,
        "returncode": result.returncode,
        "status": "OK" if result.returncode == 0 else "FAILED",
        "stderr_preview": (result.stderr or "")[:300],
    })

cli_status = pd.DataFrame(cli_status_rows)
display(cli_status)
if not cli_status["status"].eq("OK").all():
    failed = cli_status.loc[cli_status["status"] != "OK", "command"].tolist()
    raise RuntimeError(f"Missing or failing native CLI commands: {failed}")

module_names = [
    "src.research.walk_forward",
    "src.research.splits",
    "src.research.compare",
    "src.research.promotion",
    "src.research.robustness.runner",
    "src.research.robustness.walk_forward_efficiency",
]

module_status_rows = []
for module_name in module_names:
    try:
        __import__(module_name)
        module_status_rows.append({"module": module_name, "status": "OK", "error": ""})
    except Exception as exc:
        module_status_rows.append({"module": module_name, "status": "WARN", "error": repr(exc)[:500]})

module_status = pd.DataFrame(module_status_rows)
display(module_status)


## 6. Initialize or attach Fintech project/session


In [ ]:
fintech_init_cmd = [
    "fintech-init-project",
    "--root", FINTECH_ROOT.as_posix(),
    "--session-name", FINTECH_SESSION_NAME,
    "--with-session",
    "--colab-profile",
]

print("Fintech init command:")
print(" ".join(fintech_init_cmd))

result = subprocess.run(fintech_init_cmd, text=True, capture_output=True)
print("STDOUT:")
print(result.stdout)
if result.stderr:
    print("STDERR:")
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f"fintech-init-project failed with return code {result.returncode}")

session_manifest_candidates = sorted(
    (FINTECH_ROOT / "artifacts" / "sessions").glob("*/session_manifest.json"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if FINTECH_SESSION_ID_OVERRIDE:
    FINTECH_SESSION_ID = FINTECH_SESSION_ID_OVERRIDE
elif session_manifest_candidates:
    FINTECH_SESSION_ID = session_manifest_candidates[0].parent.name
else:
    raise FileNotFoundError("No Fintech session manifest found after initialization.")

MARKETLAKE_ROOT = FINTECH_ROOT / "data" / "curated"
DAILY_BARS_ROOT = MARKETLAKE_ROOT / "daily_bars"

FINTECH_ARCHIVE_ID = f"curated-data-{FINTECH_SESSION_ID}"
FINTECH_DRIVE_SESSION_ROOT = FINTECH_DRIVE_ROOT / "sessions" / FINTECH_SESSION_ID
FINTECH_DRIVE_ARCHIVE_ROOT = FINTECH_DRIVE_SESSION_ROOT / "archives"

print("FINTECH_SESSION_ID:", FINTECH_SESSION_ID)
print("MARKETLAKE_ROOT:", MARKETLAKE_ROOT.as_posix())
print("DAILY_BARS_ROOT:", DAILY_BARS_ROOT.as_posix())
print("FINTECH_DRIVE_ARCHIVE_ROOT:", FINTECH_DRIVE_ARCHIVE_ROOT.as_posix())


## 7. Initialize or attach StratLake session


In [ ]:
stratlake_init_cmd = [
    "stratlake-init-session",
    "--root", STRATLAKE_ROOT.as_posix(),
    "--project-name", STRATLAKE_SESSION_NAME,
    "--marketlake-root", MARKETLAKE_ROOT.as_posix(),
    "--drive-root", DRIVE_ROOT.as_posix(),
    "--enable-drive-persistence",
    "--notebook-configs",
]

print("StratLake init command:")
print(" ".join(stratlake_init_cmd))

result = subprocess.run(stratlake_init_cmd, text=True, capture_output=True)
print("STDOUT:")
print(result.stdout)
if result.stderr:
    print("STDERR:")
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f"stratlake-init-session failed with return code {result.returncode}")

STRATLAKE_SESSION_ID = STRATLAKE_SESSION_ID_OVERRIDE or STRATLAKE_SESSION_NAME
STRATLAKE_ARCHIVE_ID = f"stratlake-session-{STRATLAKE_SESSION_ID}"
STRATLAKE_DRIVE_SESSION_ROOT = STRATLAKE_DRIVE_ROOT / "sessions" / STRATLAKE_SESSION_ID
STRATLAKE_DRIVE_ARCHIVE_ROOT = STRATLAKE_DRIVE_SESSION_ROOT / "archives"
STRATLAKE_ARCHIVE_PACK_DIR = STRATLAKE_DRIVE_ARCHIVE_ROOT / STRATLAKE_ARCHIVE_ID

NOTEBOOK10_ARCHIVE_ID = f"notebook-10-walk-forward-promotion-{STRATLAKE_SESSION_ID}"
NOTEBOOK10_ARCHIVE_PACK_DIR = STRATLAKE_DRIVE_ARCHIVE_ROOT / NOTEBOOK10_ARCHIVE_ID

print("STRATLAKE_SESSION_ID:", STRATLAKE_SESSION_ID)
print("Source archive pack dir:", STRATLAKE_ARCHIVE_PACK_DIR.as_posix())
print("Notebook 10 archive pack dir:", NOTEBOOK10_ARCHIVE_PACK_DIR.as_posix())


## 8. Restore StratLake archive from Notebook 08/09

Archive restore is manual and off by default in committed source. Review the command preview, confirm the Notebook 08/09 archive pack exists in the configured Drive folder, and only then set RUN_STRATLAKE_ARCHIVE_RESTORE = True for a live runtime restore.


In [ ]:
RUN_STRATLAKE_ARCHIVE_RESTORE = False

print("STRATLAKE_ROOT:", STRATLAKE_ROOT.as_posix())
print("STRATLAKE_ARCHIVE_PACK_DIR:", STRATLAKE_ARCHIVE_PACK_DIR.as_posix())
print("Archive pack exists:", STRATLAKE_ARCHIVE_PACK_DIR.exists())

restore_cmd = [
    "stratlake-session-archive-restore-bootstrap",
    "--archive-root", STRATLAKE_ARCHIVE_PACK_DIR.as_posix(),
    "--target-root", ".",
    "--validate-before-restore",
    "--inspect-before-restore",
    "--overwrite-policy", "overwrite_allowed",
]

print("Restore command preview:")
print(" ".join(restore_cmd))

if RUN_STRATLAKE_ARCHIVE_RESTORE:
    if not STRATLAKE_ARCHIVE_PACK_DIR.exists():
        raise FileNotFoundError(
            "Expected StratLake archive pack was not found. "
            "Run Notebook 08/09 archive checkpoint first or update STRATLAKE_SESSION_ID_OVERRIDE. "
            f"Missing: {STRATLAKE_ARCHIVE_PACK_DIR.as_posix()}"
        )

    os.chdir(STRATLAKE_ROOT)
    print("Current working directory:", Path.cwd().as_posix())
    result = subprocess.run(restore_cmd, cwd=STRATLAKE_ROOT, text=True, capture_output=True)
    print("\nSTDOUT:")
    print(result.stdout)
    if result.stderr:
        print("\nSTDERR:")
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"StratLake archive restore failed with return code {result.returncode}")
    print("StratLake archive restore completed.")
else:
    print("Manual restore is off by default in committed source. Set RUN_STRATLAKE_ARCHIVE_RESTORE=True only after reviewing the command preview and confirming the archive pack path.")


## 9. Verify restored inputs and prior artifacts


In [ ]:
restored_required_paths = [
    STRATLAKE_ROOT / "configs",
    STRATLAKE_ROOT / "configs" / "strategies.yml",
    STRATLAKE_ROOT / "artifacts",
    MARKETLAKE_ROOT,
]

restore_check_rows = []
for path in restored_required_paths:
    restore_check_rows.append({
        "path": path.as_posix(),
        "exists": path.exists(),
        "is_dir": path.is_dir(),
        "size_bytes": path.stat().st_size if path.exists() and path.is_file() else None,
    })

restore_checks = pd.DataFrame(restore_check_rows)
display(restore_checks)

if not restore_checks["exists"].all():
    missing = restore_checks.loc[restore_checks["exists"] == False, "path"].tolist()
    raise FileNotFoundError(f"Missing required restored paths: {missing}")

prior_artifact_roots = [STRATLAKE_ROOT / "artifacts", STRATLAKE_ROOT / "reports", STRATLAKE_ROOT / "data"]
prior_artifact_rows = []
for root in prior_artifact_roots:
    if not root.exists():
        continue
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in [".json", ".csv", ".parquet", ".md", ".html"]:
            try:
                rel = p.relative_to(STRATLAKE_ROOT).as_posix()
            except ValueError:
                rel = p.as_posix()
            prior_artifact_rows.append({
                "relative_path": rel,
                "path": p.as_posix(),
                "suffix": p.suffix.lower(),
                "size_bytes": p.stat().st_size,
                "modified_utc": datetime.fromtimestamp(p.stat().st_mtime, tz=timezone.utc).isoformat(),
            })

prior_artifact_inventory = (
    pd.DataFrame(prior_artifact_rows).drop_duplicates(subset=["path"]).sort_values(["relative_path"])
    if prior_artifact_rows else pd.DataFrame()
)

print("Prior artifact rows:", len(prior_artifact_inventory))
if not prior_artifact_inventory.empty:
    display(prior_artifact_inventory.head(200))


## 10. Discover feature columns for strategy preflight

This step helps prevent expected missing-column failures from being mixed with runtime failures.


In [ ]:
def parquet_columns(path: Path) -> list[str]:
    try:
        import pyarrow.parquet as pq
        return list(pq.ParquetFile(path).schema.names)
    except Exception:
        try:
            return list(pd.read_parquet(path).columns)
        except Exception:
            return []

feature_probe_roots = [
    STRATLAKE_ROOT / "data",
    STRATLAKE_ROOT / "features",
    STRATLAKE_ROOT / "artifacts",
    MARKETLAKE_ROOT,
]

feature_file_rows = []
available_feature_columns = set()
for root in feature_probe_roots:
    if not root.exists():
        continue
    for p in sorted(root.rglob("*.parquet")):
        cols = parquet_columns(p)
        if not cols:
            continue
        available_feature_columns.update(cols)
        try:
            rel = p.relative_to(STRATLAKE_ROOT).as_posix()
        except ValueError:
            rel = p.as_posix()
        feature_file_rows.append({
            "relative_path": rel,
            "columns": ",".join(cols[:60]),
            "column_count": len(cols),
            "size_bytes": p.stat().st_size,
        })

feature_file_inventory = pd.DataFrame(feature_file_rows)
print("Parquet files with readable schemas:", len(feature_file_inventory))
print("Available unique columns:", len(available_feature_columns))
print(sorted(list(available_feature_columns))[:120])
if not feature_file_inventory.empty:
    display(feature_file_inventory.head(100))


## 11. Discover and preflight candidate native strategies

The notebook attempts to infer required columns from `configs/strategies.yml`. It also includes a small explicit fallback map for known strategy contracts observed during the first smoke audit.


In [ ]:
strategies_config_path = STRATLAKE_ROOT / "configs" / "strategies.yml"

try:
    import yaml
except Exception:
    yaml = None

strategies_config = {}
strategy_entries = {}

if yaml is not None and strategies_config_path.exists():
    with strategies_config_path.open("r", encoding="utf-8") as f:
        strategies_config = yaml.safe_load(f) or {}

    raw = strategies_config.get("strategies") if isinstance(strategies_config, dict) else None
    if isinstance(raw, dict):
        strategy_entries = {str(k): (v or {}) for k, v in raw.items()}
    elif isinstance(raw, list):
        for item in raw:
            if isinstance(item, dict):
                name = item.get("name") or item.get("strategy")
                if name:
                    strategy_entries[str(name)] = item
    elif isinstance(strategies_config, dict):
        strategy_entries = {
            str(k): (v or {}) for k, v in strategies_config.items()
            if isinstance(v, dict) and not str(k).startswith("_")
        }

strategy_names = sorted(strategy_entries.keys())

# Explicit fallback hints from first Notebook 10 smoke audit. These are used only when
# the config itself does not expose a required-column list.
KNOWN_REQUIRED_COLUMN_HINTS = {
    "breakout": ["high", "low"],
    "residual_momentum": ["market_return"],
    "weighted_cross_section_ensemble": ["market_return"],
}

REQUIRED_COLUMN_KEYS = {
    "required_columns", "required_features", "feature_columns", "input_columns",
    "columns", "required_inputs", "features", "price_columns",
}

TECHNICAL_KEYS_TO_IGNORE = {
    "name", "strategy", "type", "class", "enabled", "params", "parameters",
    "lookback", "window", "threshold", "weight", "weights", "symbols",
    "start", "end", "frequency", "benchmark", "description",
}

def normalize_required_column(value: Any) -> list[str]:
    cols = []
    if isinstance(value, str):
        # Avoid treating strategy names/classes as columns.
        if value and re.match(r"^[A-Za-z_][A-Za-z0-9_]*$", value):
            cols.append(value)
    elif isinstance(value, (list, tuple, set)):
        for item in value:
            cols.extend(normalize_required_column(item))
    elif isinstance(value, dict):
        for k, v in value.items():
            if str(k).lower() in REQUIRED_COLUMN_KEYS:
                cols.extend(normalize_required_column(v))
    return cols

def infer_required_columns(strategy_name: str, cfg: Any) -> tuple[list[str], str]:
    found = []
    if isinstance(cfg, dict):
        for k, v in cfg.items():
            if str(k).lower() in REQUIRED_COLUMN_KEYS:
                found.extend(normalize_required_column(v))
        # Search one nested level for explicit required-column keys.
        for k, v in cfg.items():
            if str(k).lower() in TECHNICAL_KEYS_TO_IGNORE:
                continue
            if isinstance(v, dict):
                for nk, nv in v.items():
                    if str(nk).lower() in REQUIRED_COLUMN_KEYS:
                        found.extend(normalize_required_column(nv))
    source = "config" if found else "none"
    if not found and strategy_name in KNOWN_REQUIRED_COLUMN_HINTS:
        found = KNOWN_REQUIRED_COLUMN_HINTS[strategy_name]
        source = "known_hint"
    return sorted(dict.fromkeys(found)), source

if CANDIDATE_STRATEGIES_OVERRIDE:
    discovered_candidate_strategy_names = sorted(dict.fromkeys(CANDIDATE_STRATEGIES_OVERRIDE))
elif strategy_names:
    discovered_candidate_strategy_names = strategy_names
else:
    discovered_candidate_strategy_names = ["momentum_v1"]

preflight_rows = []
for strategy_name in discovered_candidate_strategy_names:
    cfg = strategy_entries.get(strategy_name, {})
    required_cols, requirement_source = infer_required_columns(strategy_name, cfg)
    missing_cols = sorted([c for c in required_cols if c not in available_feature_columns])
    preflight_rows.append({
        "strategy": strategy_name,
        "required_columns": ",".join(required_cols),
        "required_column_source": requirement_source,
        "missing_columns": ",".join(missing_cols),
        "preflight_runnable": len(missing_cols) == 0,
        "preflight_note": "missing_required_columns" if missing_cols else ("no_required_columns_inferred" if not required_cols else "ok"),
    })

strategy_preflight = pd.DataFrame(preflight_rows)
if CANDIDATE_STRATEGY_LIMIT is not None:
    strategy_preflight = strategy_preflight.head(int(CANDIDATE_STRATEGY_LIMIT)).copy()

preflight_skipped = strategy_preflight.loc[~strategy_preflight["preflight_runnable"]].copy()
preflight_runnable = strategy_preflight.loc[strategy_preflight["preflight_runnable"]].copy()
preflight_skipped_strategies = preflight_skipped["strategy"].astype(str).tolist()
preflight_skipped_details = preflight_skipped[["strategy", "missing_columns", "required_column_source", "preflight_note"]].to_dict(orient="records")

if RUN_ONLY_PREFLIGHT_RUNNABLE_STRATEGIES:
    candidate_strategy_names = preflight_runnable["strategy"].astype(str).tolist()
else:
    candidate_strategy_names = strategy_preflight["strategy"].astype(str).tolist()

print("Discovered candidate strategies:", len(discovered_candidate_strategy_names))
print("Preflight runnable strategies:", len(preflight_runnable))
print("Preflight skipped strategies:", len(preflight_skipped))
print("Strategies selected for native execution:", len(candidate_strategy_names))
if preflight_skipped_strategies:
    print("Skipped by preflight:", ", ".join(preflight_skipped_strategies))
display(strategy_preflight)

if strategies_config_path.exists():
    print("\nstrategies.yml preview:")
    print(strategies_config_path.read_text(encoding="utf-8")[:4000])
else:
    print("strategies.yml not found:", strategies_config_path.as_posix())


## 12. Run walk-forward strategy smoke/expanded evaluation

Metrics are read from native artifacts when a matching run artifact can be found. Stdout parsing remains as a fallback for early notebook staging.


In [ ]:
def extract(pattern: str, text: str, default=None, cast=None):
    match = re.search(pattern, text, flags=re.MULTILINE)
    if not match:
        return default
    value = match.group(1).strip()
    if cast is None:
        return value
    try:
        return cast(value)
    except Exception:
        return default

def extract_percent(pattern: str, text: str, default=None):
    value = extract(pattern, text, default=default, cast=float)
    if value is None:
        return default
    return value / 100.0

def parse_strategy_stdout(strategy_name: str, window: dict[str, str], stdout: str, stderr: str, returncode: int) -> dict[str, Any]:
    return {
        "strategy": extract(r"^strategy:\s*(.+)$", stdout) or strategy_name,
        "window_name": window["window_name"],
        "analysis_start": window["start"],
        "analysis_end": window["end"],
        "run_id": extract(r"^run_id:\s*(.+)$", stdout),
        "completed": returncode == 0,
        "returncode": returncode,
        "cumulative_return": extract(r"^cumulative_return:\s*([-+0-9.]+)", stdout, cast=float),
        "sharpe_ratio": extract(r"^sharpe_ratio:\s*([-+0-9.]+)", stdout, cast=float),
        "long_pct": extract_percent(r"- long:\s*([-+0-9.]+)%", stdout),
        "short_pct": extract_percent(r"short:\s*([-+0-9.]+)%", stdout),
        "flat_pct": extract_percent(r"flat:\s*([-+0-9.]+)%", stdout),
        "trades": extract(r"- trades:\s*([0-9]+)", stdout, cast=int),
        "turnover": extract(r"turnover:\s*([-+0-9.]+)", stdout, cast=float),
        "avg_holding_bars": extract(r"- avg holding:\s*([-+0-9.]+)\s*bars", stdout, cast=float),
        "qa_status": extract(r"- status:\s*(.+)$", stdout),
        "qa_rows": extract(r"- rows:\s*([0-9]+)", stdout, cast=int),
        "qa_symbols": extract(r"symbols:\s*([0-9]+)", stdout, cast=int),
        "benchmark_return": extract_percent(r"- benchmark return:\s*([-+0-9.]+)%", stdout),
        "excess_return": extract_percent(r"- excess return:\s*([-+0-9.]+)%", stdout),
        "correlation": extract(r"- correlation:\s*([-+0-9.]+)", stdout, cast=float),
    }

METRIC_ALIASES = {
    "cumulative_return": ["cumulative_return", "total_return", "strategy_return"],
    "sharpe_ratio": ["sharpe_ratio", "sharpe"],
    "benchmark_return": ["benchmark_return"],
    "excess_return": ["excess_return", "active_return"],
    "turnover": ["turnover"],
    "trades": ["trades", "trade_count", "n_trades"],
    "correlation": ["correlation", "benchmark_correlation"],
}

def flatten_dict(obj: Any, prefix: str = "") -> dict[str, Any]:
    out = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            key = f"{prefix}.{k}" if prefix else str(k)
            out.update(flatten_dict(v, key))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            key = f"{prefix}.{i}" if prefix else str(i)
            out.update(flatten_dict(v, key))
    else:
        out[prefix] = obj
    return out

def coerce_float(value: Any):
    if value is None or value == "":
        return None
    try:
        return float(value)
    except Exception:
        return None

def find_metric_in_flat(flat: dict[str, Any], aliases: list[str]):
    lower_items = {k.lower(): v for k, v in flat.items()}
    for alias in aliases:
        alias_l = alias.lower()
        # exact ending match handles paths like metrics.cumulative_return.
        for k, v in lower_items.items():
            if k == alias_l or k.endswith("." + alias_l):
                return v
    return None

def load_artifact_metrics_for_run(run_id: str | None) -> tuple[dict[str, Any], str]:
    if not run_id:
        return {}, "stdout"
    search_roots = [STRATLAKE_ROOT / "artifacts", STRATLAKE_ROOT / "reports", STRATLAKE_ROOT / "data"]
    candidates = []
    for root in search_roots:
        if not root.exists():
            continue
        for p in root.rglob("*"):
            if p.is_file() and run_id in p.as_posix() and p.suffix.lower() in [".json", ".csv"]:
                candidates.append(p)
    for p in sorted(candidates, key=lambda x: (x.suffix.lower() != ".json", len(x.as_posix()))):
        try:
            if p.suffix.lower() == ".json":
                payload = json.loads(p.read_text(encoding="utf-8"))
                flat = flatten_dict(payload)
                metrics = {}
                for canonical, aliases in METRIC_ALIASES.items():
                    v = find_metric_in_flat(flat, aliases)
                    if v is not None:
                        metrics[canonical] = coerce_float(v) if canonical != "trades" else int(float(v))
                if metrics:
                    metrics["artifact_metric_path"] = p.as_posix()
                    return metrics, "artifact_json"
            elif p.suffix.lower() == ".csv":
                df = pd.read_csv(p)
                if len(df) > 0:
                    row = df.iloc[-1].to_dict()
                    flat = {str(k): v for k, v in row.items()}
                    metrics = {}
                    for canonical, aliases in METRIC_ALIASES.items():
                        v = find_metric_in_flat(flat, aliases)
                        if v is not None:
                            metrics[canonical] = coerce_float(v) if canonical != "trades" else int(float(v))
                    if metrics:
                        metrics["artifact_metric_path"] = p.as_posix()
                        return metrics, "artifact_csv"
        except Exception:
            continue
    return {}, "stdout"

def classify_stderr(stderr: str, stdout: str, returncode: int) -> tuple[str, str]:
    """Classify native command warnings without collapsing everything into one broad bucket.

    The goal is audit readability, not perfect root-cause attribution. Categories are intentionally
    conservative and stable for notebook review artifacts.
    """
    raw_text = "\n".join([stderr or "", stdout or ""])
    text = raw_text.lower()
    categories: list[str] = []

    def has_any(*patterns: str) -> bool:
        return any(pattern in text for pattern in patterns)

    if returncode != 0:
        categories.append("runtime_failed")

    if has_any("missing required column", "missing columns", "missing column") or ("missing" in text and "column" in text):
        categories.append("missing_required_columns")

    if "pct_long + pct_short + pct_flat" in text or ("pct" in text and "sum" in text):
        categories.append("signal_pct_consistency")

    if "qa summary" in text and "warn" in text:
        categories.append("qa_warn")
    elif "qa" in text and "warn" in text:
        categories.append("qa_warn")

    # Degenerate benchmark/strategy diagnostics: useful warnings, but not the same as a runtime failure.
    if has_any("benchmark buy-and-hold", "buy-and-hold", "buyandholdstrategy", "benchmark strategy") and has_any("always long", "no trades", "degenerate"):
        categories.append("benchmark_degenerate_warning")
    if has_any("strategy appears flat", "flat strategy", "no strategy trades", "zero trades", "always flat", "no trades were generated"):
        categories.append("strategy_degenerate_warning")
    if has_any("constant input", "flat return", "flat series", "invalid value encountered in divide", "invalid value encountered in scalar divide"):
        categories.append("flat_series_correlation_warning")

    # Keep generic numeric warnings only when they have not already been classified into a clearer bucket.
    numeric_terms = ["runtimewarning", "invalid value", "divide by zero", "overflow encountered", "underflow encountered"]
    if has_any(*numeric_terms) and not any(
        c in categories for c in [
            "benchmark_degenerate_warning",
            "strategy_degenerate_warning",
            "flat_series_correlation_warning",
        ]
    ):
        categories.append("numeric_runtime_warning")

    if has_any("traceback", "exception") or ("error" in text and returncode != 0):
        categories.append("exception_or_error_text")

    if stderr and not categories:
        categories.append("stderr_other")

    # Remove duplicates while preserving stable sorted output for diffs.
    categories = sorted(dict.fromkeys(categories))

    severity = "ok"
    error_categories = {"runtime_failed", "exception_or_error_text", "missing_required_columns"}
    if any(category in error_categories for category in categories):
        severity = "error"
    elif categories:
        severity = "warn"
    return severity, ",".join(categories)

walk_forward_rows = []
walk_forward_logs = {}

os.chdir(STRATLAKE_ROOT)
print("Current working directory:", Path.cwd().as_posix())

if not candidate_strategy_names:
    print("No strategies selected for native execution after preflight.")

for strategy_name in candidate_strategy_names:
    for window in WALK_FORWARD_WINDOWS:
        cmd = [
            "stratlake-run-strategy",
            "--strategies-config", "configs/strategies.yml",
            "--strategy", strategy_name,
            "--start", window["start"],
            "--end", window["end"],
        ]

        print("\nNative walk-forward command:")
        print(" ".join(cmd))

        if RUN_NATIVE_WALK_FORWARD_EVALUATION:
            result = subprocess.run(cmd, cwd=STRATLAKE_ROOT, text=True, capture_output=True)
            stdout = result.stdout or ""
            stderr = result.stderr or ""
            print("STDOUT:")
            print(stdout)
            if stderr:
                print("STDERR:")
                print(stderr)
            print("Return code:", result.returncode)

            parsed = parse_strategy_stdout(strategy_name, window, stdout, stderr, result.returncode)
            artifact_metrics, metric_source = load_artifact_metrics_for_run(parsed.get("run_id"))
            for k, v in artifact_metrics.items():
                if k != "artifact_metric_path" and v is not None:
                    parsed[k] = v
            parsed["metric_source"] = metric_source
            parsed["artifact_metric_path"] = artifact_metrics.get("artifact_metric_path", "")
            severity, categories = classify_stderr(stderr, stdout, result.returncode)
            parsed["warning_severity"] = severity
            parsed["warning_categories"] = categories
            parsed["stderr_warning"] = stderr.strip() if stderr else ""
            parsed["stdout_preview"] = stdout[:1000]
            parsed["stderr_preview"] = stderr[:1000]

            log_key = f"{strategy_name}::{window['window_name']}"
            walk_forward_logs[log_key] = {"stdout": stdout, "stderr": stderr, "returncode": result.returncode}
            walk_forward_rows.append(parsed)
        else:
            print("Dry run only; not executing native strategy command.")

walk_forward_results = pd.DataFrame(walk_forward_rows)

# Add preflight details back onto results.
if not walk_forward_results.empty and not strategy_preflight.empty:
    walk_forward_results = walk_forward_results.merge(
        strategy_preflight[["strategy", "required_columns", "missing_columns", "preflight_note"]],
        on="strategy",
        how="left",
    )

if walk_forward_results.empty:
    print("No walk-forward rows were produced.")
else:
    display_cols = [
        "strategy", "window_name", "completed", "returncode", "metric_source", "warning_severity",
        "warning_categories", "qa_status", "qa_rows", "qa_symbols", "cumulative_return",
        "benchmark_return", "excess_return", "sharpe_ratio", "trades", "turnover", "correlation",
        "required_columns", "missing_columns", "artifact_metric_path",
    ]
    display(walk_forward_results[[c for c in display_cols if c in walk_forward_results.columns]])


## 13. Add diagnostic flags for financial interpretation

A strategy can show positive excess return simply by staying flat while the benchmark declines. This cell makes that condition explicit.


In [ ]:
def add_financial_diagnostic_flags(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    out = df.copy()
    for col in ["cumulative_return", "benchmark_return", "excess_return", "trades", "turnover", "long_pct", "short_pct", "flat_pct"]:
        if col not in out.columns:
            out[col] = pd.NA
    out["is_flat_or_inactive"] = (
        out["cumulative_return"].fillna(0).abs().lt(1e-12)
        & out["trades"].fillna(0).astype(float).le(0)
    )
    out["benchmark_avoidance_outperformance"] = (
        out["is_flat_or_inactive"]
        & out["benchmark_return"].fillna(0).astype(float).lt(0)
        & out["excess_return"].fillna(0).astype(float).gt(0)
    )

    out["high_turnover_warning"] = pd.to_numeric(out["turnover"], errors="coerce").fillna(0).gt(0.50)
    out["active_negative_return"] = (
        pd.to_numeric(out["trades"], errors="coerce").fillna(0).gt(0)
        & pd.to_numeric(out["cumulative_return"], errors="coerce").fillna(0).lt(0)
    )

    pct_sum = out[["long_pct", "short_pct", "flat_pct"]].astype(float).sum(axis=1, skipna=True)
    out["signal_pct_sum"] = pct_sum
    out["signal_pct_sum_warning"] = pct_sum.notna() & pct_sum.gt(0) & pct_sum.sub(1.0).abs().gt(0.05)
    return out

walk_forward_results = add_financial_diagnostic_flags(walk_forward_results)

if walk_forward_results.empty:
    print("No walk-forward results to diagnose.")
else:
    diagnostic_cols = [
        "strategy", "window_name", "completed", "cumulative_return", "benchmark_return", "excess_return",
        "trades", "turnover", "is_flat_or_inactive", "benchmark_avoidance_outperformance",
        "signal_pct_sum", "signal_pct_sum_warning", "high_turnover_warning", "active_negative_return", "warning_categories",
    ]
    display(walk_forward_results[[c for c in diagnostic_cols if c in walk_forward_results.columns]])

    caution_count = int(walk_forward_results.get("benchmark_avoidance_outperformance", pd.Series(dtype=bool)).fillna(False).sum())
    if caution_count:
        print(f"CAUTION: {caution_count} row(s) show positive excess return from flat/inactive benchmark avoidance. Do not treat these as alpha evidence.")


## 14. Build robustness summary from split-level results


In [ ]:
def safe_mean(series):
    series = pd.to_numeric(series, errors="coerce").dropna()
    return float(series.mean()) if len(series) else None

def safe_min(series):
    series = pd.to_numeric(series, errors="coerce").dropna()
    return float(series.min()) if len(series) else None

def safe_max(series):
    series = pd.to_numeric(series, errors="coerce").dropna()
    return float(series.max()) if len(series) else None

def safe_std(series):
    series = pd.to_numeric(series, errors="coerce").dropna()
    return float(series.std(ddof=0)) if len(series) else None

def count_true(series):
    return int(series.fillna(False).astype(bool).sum()) if len(series) else 0

if walk_forward_results.empty:
    robustness_summary = pd.DataFrame()
    print("No walk-forward results available for robustness summary.")
else:
    summary_rows = []
    required_windows = len(WALK_FORWARD_WINDOWS)
    for strategy, group in walk_forward_results.groupby("strategy", dropna=False):
        completed_group = group[group["completed"] == True].copy()
        completed_windows = int(completed_group.shape[0])

        excess = completed_group.get("excess_return", pd.Series(dtype=float))
        cumulative = completed_group.get("cumulative_return", pd.Series(dtype=float))
        sharpe = completed_group.get("sharpe_ratio", pd.Series(dtype=float))
        turnover = completed_group.get("turnover", pd.Series(dtype=float))
        trades = completed_group.get("trades", pd.Series(dtype=float))

        warning_categories = sorted(set(
            cat for value in group.get("warning_categories", pd.Series(dtype=str)).fillna("").astype(str)
            for cat in value.split(",") if cat
        ))

        summary_rows.append({
            "strategy": strategy,
            "required_windows": required_windows,
            "completed_windows": completed_windows,
            "completion_rate": completed_windows / required_windows if required_windows else None,
            "positive_excess_windows": int((pd.to_numeric(excess, errors="coerce") > 0).sum()) if len(excess) else 0,
            "positive_excess_rate": float((pd.to_numeric(excess, errors="coerce") > 0).mean()) if len(excess) else None,
            "positive_cumulative_windows": int((pd.to_numeric(cumulative, errors="coerce") > 0).sum()) if len(cumulative) else 0,
            "positive_cumulative_rate": float((pd.to_numeric(cumulative, errors="coerce") > 0).mean()) if len(cumulative) else None,
            "mean_excess_return": safe_mean(excess),
            "min_excess_return": safe_min(excess),
            "max_excess_return": safe_max(excess),
            "std_excess_return": safe_std(excess),
            "mean_cumulative_return": safe_mean(cumulative),
            "min_cumulative_return": safe_min(cumulative),
            "mean_sharpe_ratio": safe_mean(sharpe),
            "min_sharpe_ratio": safe_min(sharpe),
            "mean_turnover": safe_mean(turnover),
            "max_turnover": safe_max(turnover),
            "total_trades": int(pd.to_numeric(trades, errors="coerce").fillna(0).sum()) if len(trades) else 0,
            "qa_clean_windows": int(completed_group.get("qa_status", pd.Series(dtype=str)).fillna("").str.lower().isin(["ok", "pass", "passed", "clean"]).sum()),
            "warning_windows": int(group.get("warning_severity", pd.Series(dtype=str)).fillna("").isin(["warn", "error"]).sum()),
            "error_windows": int(group.get("warning_severity", pd.Series(dtype=str)).fillna("").eq("error").sum()),
            "flat_or_inactive_windows": count_true(completed_group.get("is_flat_or_inactive", pd.Series(dtype=bool))),
            "benchmark_avoidance_outperformance_windows": count_true(completed_group.get("benchmark_avoidance_outperformance", pd.Series(dtype=bool))),
            "signal_pct_sum_warning_windows": count_true(completed_group.get("signal_pct_sum_warning", pd.Series(dtype=bool))),
            "high_turnover_warning_windows": count_true(completed_group.get("high_turnover_warning", pd.Series(dtype=bool))),
            "active_negative_return_windows": count_true(completed_group.get("active_negative_return", pd.Series(dtype=bool))),
            "warning_categories": ",".join(warning_categories),
        })

    robustness_summary = pd.DataFrame(summary_rows)
    sort_cols = [c for c in ["completion_rate", "positive_cumulative_rate", "positive_excess_rate", "mean_cumulative_return", "mean_excess_return"] if c in robustness_summary.columns]
    if sort_cols:
        robustness_summary = robustness_summary.sort_values(sort_cols, ascending=[False] * len(sort_cols))
    display(robustness_summary)


## 15. Plot walk-forward robustness diagnostics


In [ ]:
if walk_forward_results.empty:
    print("No walk-forward results available to plot.")
else:
    metric_cols = ["excess_return", "cumulative_return", "benchmark_return", "sharpe_ratio", "turnover", "trades"]
    available_metrics = [c for c in metric_cols if c in walk_forward_results.columns]

    for metric in available_metrics:
        plot_df = walk_forward_results.dropna(subset=[metric]).copy()
        if plot_df.empty:
            print(f"No values available for {metric}.")
            continue

        pivot = plot_df.pivot_table(index="window_name", columns="strategy", values=metric, aggfunc="mean")
        ax = pivot.plot(kind="bar", title=f"Walk-forward robustness: {metric}", figsize=(12, 4))
        ax.set_xlabel("window")
        ax.set_ylabel(metric)
        ax.axhline(0, linewidth=1)
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.show()

    diagnostic_cols = [
        "strategy", "window_name", "run_id", "completed", "returncode", "metric_source", "qa_status",
        "qa_rows", "qa_symbols", "trades", "turnover", "avg_holding_bars",
        "long_pct", "short_pct", "flat_pct", "signal_pct_sum_warning",
        "benchmark_avoidance_outperformance", "warning_severity", "warning_categories",
    ]
    display(walk_forward_results[[c for c in diagnostic_cols if c in walk_forward_results.columns]])


## 16. Apply improved notebook-level promotion gates

These gates are intentionally conservative. In smoke mode, a successful outcome is usually **no promoted strategies** and a clear set of surfaced warnings, skipped strategy contracts, and review reasons.


In [ ]:
PROMOTION_GATES = {
    "min_completion_rate": 1.0 if NOTEBOOK10_MODE == "smoke" else 0.80,
    "min_positive_excess_rate": 0.60 if NOTEBOOK10_MODE != "smoke" else 1.0,
    "min_positive_cumulative_rate": 0.60 if NOTEBOOK10_MODE != "smoke" else 1.0,
    "min_mean_excess_return": 0.0,
    "min_mean_cumulative_return": 0.0,
    "min_mean_sharpe_ratio": 0.0,
    "min_total_trades": 3,
    "max_mean_turnover": 0.50,
    "require_any_qa_clean_window": True,
    "allow_warning_windows": False,
    "allow_error_windows": False,
    "allow_benchmark_avoidance_outperformance": False,
    "allow_signal_pct_sum_warning": False,
}

def classify_strategy(row: pd.Series) -> tuple[str, list[str]]:
    reasons = []

    def below(name: str, gate: str):
        value = row.get(name)
        threshold = PROMOTION_GATES[gate]
        return pd.isna(value) or value < threshold

    if below("completion_rate", "min_completion_rate"):
        reasons.append("incomplete_walk_forward_windows")
    if below("positive_excess_rate", "min_positive_excess_rate"):
        reasons.append("insufficient_positive_excess_consistency")
    if below("positive_cumulative_rate", "min_positive_cumulative_rate"):
        reasons.append("insufficient_positive_cumulative_consistency")
    if below("mean_excess_return", "min_mean_excess_return"):
        reasons.append("non_positive_mean_excess_return")
    if below("mean_cumulative_return", "min_mean_cumulative_return"):
        reasons.append("non_positive_mean_cumulative_return")
    if below("mean_sharpe_ratio", "min_mean_sharpe_ratio"):
        reasons.append("non_positive_mean_sharpe")
    if below("total_trades", "min_total_trades"):
        reasons.append("insufficient_trade_activity")

    mean_turnover = row.get("mean_turnover")
    if not pd.isna(mean_turnover) and mean_turnover > PROMOTION_GATES["max_mean_turnover"]:
        reasons.append("turnover_above_gate")

    if PROMOTION_GATES["require_any_qa_clean_window"] and int(row.get("qa_clean_windows") or 0) <= 0:
        reasons.append("no_qa_clean_windows")
    if not PROMOTION_GATES["allow_warning_windows"] and int(row.get("warning_windows") or 0) > 0:
        reasons.append("warning_windows_present")
    if not PROMOTION_GATES["allow_error_windows"] and int(row.get("error_windows") or 0) > 0:
        reasons.append("error_windows_present")
    if not PROMOTION_GATES["allow_benchmark_avoidance_outperformance"] and int(row.get("benchmark_avoidance_outperformance_windows") or 0) > 0:
        reasons.append("flat_benchmark_avoidance_outperformance")
    if not PROMOTION_GATES["allow_signal_pct_sum_warning"] and int(row.get("signal_pct_sum_warning_windows") or 0) > 0:
        reasons.append("signal_pct_sum_warning")

    completion_rate = row.get("completion_rate")
    if not reasons:
        return "promoted", []
    if completion_rate is not None and not pd.isna(completion_rate) and completion_rate > 0 and len(reasons) <= 3:
        return "watchlist", reasons
    if completion_rate is not None and not pd.isna(completion_rate) and completion_rate > 0:
        return "needs_review", reasons
    return "failed_runtime_or_qa", reasons

if robustness_summary.empty:
    promotion_review = pd.DataFrame()
    print("No robustness summary available for promotion review.")
else:
    review_rows = []
    for _, row in robustness_summary.iterrows():
        decision, reasons = classify_strategy(row)
        review_row = row.to_dict()
        review_row["promotion_decision"] = decision
        review_row["promotion_reasons"] = ", ".join(reasons) if reasons else "passed_notebook_gates"
        review_rows.append(review_row)

    promotion_review = pd.DataFrame(review_rows)
    decision_order = {"promoted": 0, "watchlist": 1, "needs_review": 2, "failed_runtime_or_qa": 3}
    promotion_review["_decision_order"] = promotion_review["promotion_decision"].map(decision_order).fillna(99)
    promotion_review = promotion_review.sort_values(
        ["_decision_order", "mean_cumulative_return", "mean_excess_return", "mean_sharpe_ratio"],
        ascending=[True, False, False, False],
    ).drop(columns=["_decision_order"])

    print("Promotion gates:")
    print(json.dumps(PROMOTION_GATES, indent=2))
    if NOTEBOOK10_MODE == "smoke":
        print("SMOKE MODE NOTE: no promoted strategies is an acceptable and expected outcome.")
    display(promotion_review)


## 17. Smoke audit interpretation and import-readiness notes

This section summarizes whether the notebook behaved as expected for smoke-mode import staging. The imported source also reports a more specific warning taxonomy so broad runtime warnings are separated into benchmark-degenerate, strategy-degenerate, flat-series correlation, signal-percentage, QA, and runtime-failure classes.

A successful smoke result usually means:

- the restored Notebook 08/09 archive was usable;
- native commands resolved;
- preflight skipped feature-incompatible strategies before execution;
- runnable strategies produced artifact-backed metrics;
- no strategies were promoted from a one-window smoke run;
- warnings and flat/inactive strategy behavior were surfaced rather than hidden.


In [ ]:
metric_source_counts = (
    walk_forward_results["metric_source"].value_counts().to_dict()
    if "walk_forward_results" in globals() and not walk_forward_results.empty and "metric_source" in walk_forward_results.columns
    else {}
)

warning_category_counts = {}
if "walk_forward_results" in globals() and not walk_forward_results.empty and "warning_categories" in walk_forward_results.columns:
    for item in walk_forward_results["warning_categories"].fillna(""):
        for category in [x.strip() for x in str(item).split(",") if x.strip()]:
            warning_category_counts[category] = warning_category_counts.get(category, 0) + 1


warning_category_definitions = {
    "runtime_failed": "Native command returned a non-zero exit status.",
    "missing_required_columns": "Strategy feature contract was not satisfied by restored data.",
    "signal_pct_consistency": "Signal long/short/flat percentages did not sum as expected or native output reported a pct consistency warning.",
    "qa_warn": "Native strategy QA reported WARN status.",
    "benchmark_degenerate_warning": "Benchmark/buy-and-hold comparator produced a degenerate diagnostic such as always-long or no-trade behavior.",
    "strategy_degenerate_warning": "Strategy behavior appeared flat, inactive, always flat, or no-trade.",
    "flat_series_correlation_warning": "Numeric warning likely caused by constant/flat series in correlation or division calculations.",
    "numeric_runtime_warning": "Generic numeric runtime warning not classified into a more specific bucket.",
    "exception_or_error_text": "Traceback/exception/error text was detected.",
    "stderr_other": "Stderr was present but did not match a known category.",
}


diagnostic_counts = {}
if "walk_forward_results" in globals() and not walk_forward_results.empty:
    for col in [
        "is_flat_or_inactive",
        "benchmark_avoidance_outperformance",
        "signal_pct_sum_warning",
        "high_turnover_warning",
        "qa_is_clean",
        "active_negative_return",
    ]:
        if col in walk_forward_results.columns:
            diagnostic_counts[col] = int(walk_forward_results[col].fillna(False).astype(bool).sum())

promotion_decision_counts = (
    promotion_review["promotion_decision"].value_counts().to_dict()
    if "promotion_review" in globals() and not promotion_review.empty and "promotion_decision" in promotion_review.columns
    else {}
)

smoke_audit_status = "not_smoke_mode"
if NOTEBOOK10_MODE == "smoke":
    promoted_count = int(promotion_decision_counts.get("promoted", 0))
    selected_count = int(len(candidate_strategy_names)) if "candidate_strategy_names" in globals() else 0
    executed_count = int(len(walk_forward_results)) if "walk_forward_results" in globals() else 0
    skipped_count = int(len(preflight_skipped)) if "preflight_skipped" in globals() else 0
    smoke_audit_status = "pass" if promoted_count == 0 and executed_count == selected_count else "review_needed"

smoke_audit_summary = {
    "notebook10_mode": NOTEBOOK10_MODE,
    "smoke_audit_status": smoke_audit_status,
    "interpretation": (
        "Smoke mode validates workflow wiring and diagnostic surfacing only; expanded mode is required before promotion-grade interpretation."
        if NOTEBOOK10_MODE == "smoke"
        else "Expanded mode provides broader evidence but still requires human review before promotion."
    ),
    "candidate_strategy_count_discovered": int(len(discovered_candidate_strategy_names)) if "discovered_candidate_strategy_names" in globals() else 0,
    "preflight_runnable_count": int(len(preflight_runnable)) if "preflight_runnable" in globals() else 0,
    "preflight_skipped_count": int(len(preflight_skipped)) if "preflight_skipped" in globals() else 0,
    "preflight_skipped_strategies": preflight_skipped_strategies if "preflight_skipped_strategies" in globals() else [],
    "native_execution_rows": int(len(walk_forward_results)) if "walk_forward_results" in globals() else 0,
    "promotion_decision_counts": promotion_decision_counts,
    "metric_source_counts": metric_source_counts,
    "warning_category_counts": warning_category_counts,
    "warning_category_definitions": warning_category_definitions,
    "diagnostic_counts": diagnostic_counts,
}

display(pd.DataFrame([smoke_audit_summary]))
print(json.dumps(smoke_audit_summary, indent=2))


## 18. Write Notebook 10 review outputs and artifact inventory


In [ ]:
NOTEBOOK10_REVIEW_DIR = STRATLAKE_ROOT / "artifacts" / "notebook_10_walk_forward_promotion_review"
NOTEBOOK10_REVIEW_DIR.mkdir(parents=True, exist_ok=True)

if "strategy_preflight" in globals() and not strategy_preflight.empty:
    strategy_preflight.to_csv(NOTEBOOK10_REVIEW_DIR / "preflight_summary.csv", index=False)
    strategy_preflight.to_json(NOTEBOOK10_REVIEW_DIR / "preflight_summary.json", orient="records", indent=2)

if "walk_forward_results" in globals() and not walk_forward_results.empty:
    walk_forward_results.to_csv(NOTEBOOK10_REVIEW_DIR / "walk_forward_results.csv", index=False)
    walk_forward_results.to_json(NOTEBOOK10_REVIEW_DIR / "walk_forward_results.json", orient="records", indent=2)

if "robustness_summary" in globals() and not robustness_summary.empty:
    robustness_summary.to_csv(NOTEBOOK10_REVIEW_DIR / "robustness_summary.csv", index=False)
    robustness_summary.to_json(NOTEBOOK10_REVIEW_DIR / "robustness_summary.json", orient="records", indent=2)

if "promotion_review" in globals() and not promotion_review.empty:
    promotion_review.to_csv(NOTEBOOK10_REVIEW_DIR / "promotion_review.csv", index=False)
    promotion_review.to_json(NOTEBOOK10_REVIEW_DIR / "promotion_review.json", orient="records", indent=2)

summary_payload = {
    "notebook": "Notebook 10 — StratLake Walk-Forward Robustness and Promotion Review",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source_archive_id": STRATLAKE_ARCHIVE_ID,
    "notebook10_archive_id": NOTEBOOK10_ARCHIVE_ID,
    "notebook10_mode": NOTEBOOK10_MODE,
    "walk_forward_windows": WALK_FORWARD_WINDOWS,
    "candidate_strategies_selected": candidate_strategy_names,
    "candidate_strategy_count_selected": len(candidate_strategy_names),
    "candidate_strategy_count_discovered": int(len(discovered_candidate_strategy_names)) if "discovered_candidate_strategy_names" in globals() else 0,
    "preflight_total_count": int(len(strategy_preflight)) if "strategy_preflight" in globals() else 0,
    "preflight_runnable_count": int(len(preflight_runnable)) if "preflight_runnable" in globals() else 0,
    "preflight_skipped_count": int(len(preflight_skipped)) if "preflight_skipped" in globals() else 0,
    "preflight_skipped_strategies": preflight_skipped_strategies if "preflight_skipped_strategies" in globals() else [],
    "preflight_skipped_details": preflight_skipped_details if "preflight_skipped_details" in globals() else [],
    "smoke_audit_summary": smoke_audit_summary if "smoke_audit_summary" in globals() else {},
    "promotion_gates": PROMOTION_GATES,
    "review_dir": NOTEBOOK10_REVIEW_DIR.as_posix(),
    "interpretation_warning": "Smoke mode validates workflow wiring only and should not be treated as promotion-grade financial evidence.",
}

if "promotion_review" in globals() and not promotion_review.empty:
    summary_payload["promotion_decision_counts"] = promotion_review["promotion_decision"].value_counts().to_dict()
    summary_payload["promoted_strategies"] = promotion_review.loc[
        promotion_review["promotion_decision"] == "promoted", "strategy"
    ].astype(str).tolist()
else:
    summary_payload["promotion_decision_counts"] = {}
    summary_payload["promoted_strategies"] = []

(NOTEBOOK10_REVIEW_DIR / "summary.json").write_text(json.dumps(summary_payload, indent=2), encoding="utf-8")
if "smoke_audit_summary" in globals():
    (NOTEBOOK10_REVIEW_DIR / "smoke_audit_summary.json").write_text(
        json.dumps(smoke_audit_summary, indent=2), encoding="utf-8"
    )

artifact_roots = [STRATLAKE_ROOT / "artifacts", STRATLAKE_ROOT / "data", STRATLAKE_ROOT / "reports"]
run_ids = []
if "walk_forward_results" in globals() and not walk_forward_results.empty and "run_id" in walk_forward_results.columns:
    run_ids = [str(x) for x in walk_forward_results["run_id"].dropna().tolist()]

artifact_rows = []
for root in artifact_roots:
    if not root.exists():
        continue
    for p in root.rglob("*"):
        if not p.is_file():
            continue
        path_text = p.as_posix()
        matched_run_ids = [run_id for run_id in run_ids if run_id and run_id in path_text]
        if matched_run_ids or NOTEBOOK10_REVIEW_DIR.as_posix() in path_text or p.suffix.lower() in [".json", ".csv", ".parquet", ".md", ".html"]:
            try:
                rel = p.relative_to(STRATLAKE_ROOT).as_posix()
            except ValueError:
                rel = p.as_posix()
            artifact_rows.append({
                "matched_run_ids": ",".join(matched_run_ids),
                "relative_path": rel,
                "path": p.as_posix(),
                "suffix": p.suffix,
                "size_bytes": p.stat().st_size,
                "modified_utc": datetime.fromtimestamp(p.stat().st_mtime, tz=timezone.utc).isoformat(),
            })

artifact_inventory = (
    pd.DataFrame(artifact_rows).drop_duplicates(subset=["path"]).sort_values(["matched_run_ids", "relative_path"])
    if artifact_rows else pd.DataFrame()
)

if not artifact_inventory.empty:
    artifact_inventory.to_csv(NOTEBOOK10_REVIEW_DIR / "artifact_inventory.csv", index=False)
    artifact_inventory.to_json(NOTEBOOK10_REVIEW_DIR / "artifact_inventory.json", orient="records", indent=2)

print("Notebook 10 review dir:", NOTEBOOK10_REVIEW_DIR.as_posix())
print("Artifact rows:", len(artifact_inventory))
if not artifact_inventory.empty:
    display(artifact_inventory.head(250))


## 19. Optional archive checkpoint after Notebook 10 review

Default is `False` so the first smoke-mode review does not create a large new archive before the outputs are inspected.


In [ ]:
RUN_STRATLAKE_ARCHIVE_CHECKPOINT = False

archive_cmd = [
    "stratlake-session-archive-bootstrap",
    "--root", STRATLAKE_ROOT.as_posix(),
    "--archive-id", NOTEBOOK10_ARCHIVE_ID,
    "--archive-collision-policy", "overwrite_allowed",
    "--drive-root", STRATLAKE_DRIVE_ARCHIVE_ROOT.as_posix(),
    "--copy-policy", "overwrite_allowed",
    "--include-features",
    "--include-artifacts",
    "--include-configs",
    "--validate-after-copy",
    "--inspect-after-copy",
]

print("StratLake Notebook 10 archive checkpoint command:")
print(" ".join(archive_cmd))

if RUN_STRATLAKE_ARCHIVE_CHECKPOINT:
    result = subprocess.run(archive_cmd, cwd=STRATLAKE_ROOT, text=True, capture_output=True)
    print("\nSTDOUT:")
    print(result.stdout)
    if result.stderr:
        print("\nSTDERR:")
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"StratLake archive checkpoint failed with return code {result.returncode}")
    print("StratLake Notebook 10 archive checkpoint completed.")
else:
    print("RUN_STRATLAKE_ARCHIVE_CHECKPOINT is False; skipping archive checkpoint.")


## 20. Final handoff


In [ ]:
final_handoff = {
    "notebook": "Notebook 10 — StratLake Walk-Forward Robustness and Promotion Review",
    "source_notebooks": [
        "Notebook 08 — StratLake Strategy Backtest Artifact Review",
        "Notebook 09 — StratLake Strategy Comparison and Research Review",
    ],
    "fintech_session_id": FINTECH_SESSION_ID,
    "stratlake_session_id": STRATLAKE_SESSION_ID,
    "source_stratlake_archive_id": STRATLAKE_ARCHIVE_ID,
    "notebook10_archive_id": NOTEBOOK10_ARCHIVE_ID,
    "notebook10_mode": NOTEBOOK10_MODE,
    "analysis_start": ANALYSIS_START,
    "analysis_end": ANALYSIS_END,
    "walk_forward_windows": len(WALK_FORWARD_WINDOWS),
    "candidate_strategies_selected": len(candidate_strategy_names),
    "preflight_rows": int(len(strategy_preflight)) if "strategy_preflight" in globals() else 0,
    "preflight_runnable_count": int(len(preflight_runnable)) if "preflight_runnable" in globals() else 0,
    "preflight_skipped_count": int(len(preflight_skipped)) if "preflight_skipped" in globals() else 0,
    "preflight_skipped_strategies": preflight_skipped_strategies if "preflight_skipped_strategies" in globals() else [],
    "walk_forward_rows": int(len(walk_forward_results)) if "walk_forward_results" in globals() else 0,
    "promotion_review_rows": int(len(promotion_review)) if "promotion_review" in globals() else 0,
    "promoted_strategies": (
        promotion_review.loc[promotion_review["promotion_decision"] == "promoted", "strategy"].astype(str).tolist()
        if "promotion_review" in globals() and not promotion_review.empty else []
    ),
    "watchlist_strategies": (
        promotion_review.loc[promotion_review["promotion_decision"] == "watchlist", "strategy"].astype(str).tolist()
        if "promotion_review" in globals() and not promotion_review.empty else []
    ),
    "artifact_rows": int(len(artifact_inventory)) if "artifact_inventory" in globals() else 0,
    "smoke_audit_status": smoke_audit_summary.get("smoke_audit_status") if "smoke_audit_summary" in globals() else None,
    "metric_source_counts": smoke_audit_summary.get("metric_source_counts", {}) if "smoke_audit_summary" in globals() else {},
    "warning_category_counts": smoke_audit_summary.get("warning_category_counts", {}) if "smoke_audit_summary" in globals() else {},
    "diagnostic_counts": smoke_audit_summary.get("diagnostic_counts", {}) if "smoke_audit_summary" in globals() else {},
    "review_dir": NOTEBOOK10_REVIEW_DIR.as_posix(),
    "source_archive_pack_dir": STRATLAKE_ARCHIVE_PACK_DIR.as_posix(),
    "notebook10_archive_pack_dir": NOTEBOOK10_ARCHIVE_PACK_DIR.as_posix(),
    "next_notebook": "Notebook 11 — Alpha Sleeve, Portfolio Construction, or Research Campaign Orchestration",
    "staging_interpretation": "Smoke mode validates workflow wiring and diagnostics; expanded mode is required before any promotion-grade interpretation.",
}

display(pd.DataFrame([final_handoff]))
print(json.dumps(final_handoff, indent=2))
